In [4]:
import json
from collections import defaultdict, Counter

def analizar_campos_y_reglas(ruta_archivo):
    try:
        # 1. Abrir y leer el archivo JSON
        with open(ruta_archivo, 'r', encoding='utf-8') as archivo:
            datos = json.load(archivo)
        
        # 2. Inicializar el diccionario
        datos_agrupados = defaultdict(lambda: defaultdict(list))

        # 3. Recorrer los datos del archivo
        for elemento in datos:
            tipo = elemento.get("ruletype")
            campo = elemento.get("ruleField")
            regla = elemento.get("rule")
            
            # Convertimos la regla a string por si viene como entero
            if regla is not None:
                regla = str(regla)
            
            # Validamos que los tres campos existan (incluso si la regla es "-1" o el tipo es "none")
            if tipo and campo and regla:
                datos_agrupados[tipo][campo].append(regla)

        # 4. Mostrar los resultados
        print("=========================================")
        print("   CAMPOS Y REGLAS POR TIPO (df / none)")
        print("=========================================")
        
        for tipo, campos in datos_agrupados.items():
            print(f"\n[+] Tipo de regla: '{tipo}'")
            
            for campo, lista_reglas in campos.items():
                cantidad = len(lista_reglas)
                
                # Convertimos a set y ordenamos tratando los números negativos o strings de forma segura
                reglas_unicas = sorted(set(lista_reglas), key=lambda x: int(x) if x.replace('-', '').isdigit() else x)
                reglas_str = ", ".join(reglas_unicas)
                
                print(f"    - {campo}: {cantidad} vez/veces | Reglas usadas: [{reglas_str}]")
                
    except FileNotFoundError:
        print(f"Error: El archivo '{ruta_archivo}' no existe. Verifica la ruta.")
    except json.JSONDecodeError:
        print(f"Error: El archivo '{ruta_archivo}' no tiene un formato JSON válido.")


def analizar_por_numero_regla(ruta_archivo):
    try:
        with open(ruta_archivo, 'r', encoding='utf-8') as archivo:
            datos = json.load(archivo)
        
        info_reglas = defaultdict(lambda: {"cantidad": 0, "tipo": set(), "campos": set()})

        for elemento in datos:
            regla = elemento.get("rule")
            tipo = elemento.get("ruletype")
            campo = elemento.get("ruleField")
            
            if regla is not None:
                regla = str(regla)  # Aseguramos formato string
                info_reglas[regla]["cantidad"] += 1
                if tipo:
                    info_reglas[regla]["tipo"].add(tipo)
                if campo:
                    info_reglas[regla]["campos"].add(campo)

        print("\n=========================================")
        print("        ANÁLISIS POR NÚMERO DE REGLA      ")
        print("=========================================")
        
        # Soportamos la ordenación de números negativos como "-1"
        reglas_ordenadas = sorted(info_reglas.keys(), key=lambda x: int(x) if x.replace('-', '').isdigit() else x)
        
        for regla in reglas_ordenadas:
            datos_regla = info_reglas[regla]
            cantidad = datos_regla["cantidad"]
            tipos_str = ", ".join(datos_regla["tipo"])
            campos_str = ", ".join(datos_regla["campos"])
            
            print(f"\n[+] Regla {regla}:")
            print(f"    - Veces utilizada: {cantidad}")
            print(f"    - Tipo (ruletype): {tipos_str}")
            print(f"    - Campos afectados: {campos_str}")
                
    except FileNotFoundError:
        print(f"Error: El archivo '{ruta_archivo}' no existe.")
    except json.JSONDecodeError:
        print(f"Error: El archivo '{ruta_archivo}' no es un JSON válido.")


def analizar_frecuencia_campos(ruta_archivo):
    try:
        with open(ruta_archivo, 'r', encoding='utf-8') as archivo:
            datos = json.load(archivo)
        
        if isinstance(datos, list):
            total_reglas = len(datos)
            
            # Corregido: 'ruletype' en lugar de 'ruleType' para coincidir con tu JSON
            tipos_count = Counter(regla.get('ruletype') for regla in datos if 'ruletype' in regla)
            campos_por_tipo = defaultdict(Counter)
            
            for regla in datos:
                tipo = regla.get('ruletype')
                campo = regla.get('ruleField')
                if tipo and campo:
                    campos_por_tipo[tipo][campo] += 1
            
            print("\n" + "-" * 60)
            print(f"Total de objetos procesados: {total_reglas}")
            print("-" * 60)
            print("Desglose por tipo de regla y frecuencia de los campos:\n")
            
            for tipo, cantidad in tipos_count.items():
                print(f"[ Tipo '{tipo}' ] -> Aparece en {cantidad} elementos en total.")
                print("  Campos afectados (ordenados por frecuencia):")
                
                campos_ordenados = sorted(campos_por_tipo[tipo].items(), key=lambda x: (-x[1], x[0]))
                
                for campo, frecuencia in campos_ordenados:
                    palabra = "vez" if frecuencia == 1 else "veces"
                    print(f"    • {campo}: afectado {frecuencia} {palabra}")
                print()
                
            print("-" * 60)
        else:
            print("Error: El formato del JSON no es una lista.")

    except FileNotFoundError:
        print(f"Error: No se encontró el archivo '{ruta_archivo}'.")
    except json.JSONDecodeError:
        print(f"Error: El archivo '{ruta_archivo}' no contiene un formato JSON válido.")


# --- EJECUCIÓN DEL PROGRAMA ---
nombre_archivo = "../textos/datasetsantolimpio3.json"

# Ahora puedes ejecutar las tres funciones de manera segura con tus nuevos objetos planos
analizar_campos_y_reglas(nombre_archivo)
analizar_por_numero_regla(nombre_archivo)
analizar_frecuencia_campos(nombre_archivo)

   CAMPOS Y REGLAS POR TIPO (df / none)

[+] Tipo de regla: 'df'
    - hasTypeInc: 2 vez/veces | Reglas usadas: [4]
    - hasSupportGroup: 7 vez/veces | Reglas usadas: [8, 10, 28]
    - hasTechnician: 3 vez/veces | Reglas usadas: [24, 31]
    - incident_hasOrigin: 2 vez/veces | Reglas usadas: [30]

[+] Tipo de regla: 'noValid'
    - hasTechnician: 5 vez/veces | Reglas usadas: [38, 41]
    - hasSupportGroup: 2 vez/veces | Reglas usadas: [40]
    - incident_hasOrigin: 2 vez/veces | Reglas usadas: [39]
    - hasTypeInc: 3 vez/veces | Reglas usadas: [34]

[+] Tipo de regla: 'both'
    - hasTechnician: 2 vez/veces | Reglas usadas: [1]
    - hasTypeInc: 2 vez/veces | Reglas usadas: [2]
    - incident_hasOrigin: 2 vez/veces | Reglas usadas: [5]
    - hasSupportGroup: 1 vez/veces | Reglas usadas: [32]

[+] Tipo de regla: 'none'
    - hasTypeInc: 5 vez/veces | Reglas usadas: [-1]
    - incident_hasOrigin: 5 vez/veces | Reglas usadas: [-1]
    - hasSupportGroup: 5 vez/veces | Reglas usadas: [-1]